# Práctica 2. ChromaDB paso a paso

En la práctica 1 el vector vivía en una variable y se perdía al cerrar el cuaderno. Una base de datos lo guarda para poder buscarlo después.

ChromaDB es la base que usaremos en DocIA+. Aquí trabaja dentro de Colab. Primero en memoria, y al final en una carpeta.

Cada registro guarda cuatro cosas, y cada una tiene un motivo:

- `ids`: identificador único, para actualizar o borrar ese registro y no otro
- `documents`: el texto del trozo, porque el vector no se puede leer ni citar
- `embeddings`: la lista de números, calculada por nosotros con el mismo modelo de antes
- `metadatas`: datos para filtrar, por ejemplo la categoría. No forman parte del vector

Si solo pasáis el texto y no el vector, ChromaDB llama a un modelo que no veis. Puede no ser el de 384 números. En este cuaderno no se hace así.

Las frases siguen siendo de ejercicio.


## Instalar y cargar el mismo modelo

Hace falta la librería de la base (`chromadb`) y otra vez el modelo de la práctica 1.

Se imprime la dimensión. Tiene que salir 384. Si no sale 384, este cuaderno y el anterior no están en el mismo mapa y las distancias no significan nada.


In [ ]:
!pip install chromadb sentence-transformers -q

from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer("all-MiniLM-L6-v2")
print("Modelo listo. Dimensiones:", modelo.get_sentence_embedding_dimension())


## Crear la colección

En SQL, antes de insertar filas se crea la tabla. Aquí se crea la colección: el sitio donde vivirán los registros.

`EphemeralClient` la guarda solo en memoria. Al cerrar el cuaderno desaparece. Más adelante veréis la que se escribe en disco.

`hnsw:space = cosine` le dice cómo medir. Hoy basta con esto: en esta colección, una distancia más pequeña significa más parecido. El coseno se calcula a mano en el tema 3.

Si la colección ya existía de una ejecución anterior, se borra y se crea vacía. Así no se mezclan pruebas viejas.


In [ ]:
import chromadb

cliente = chromadb.EphemeralClient()
existentes = [c.name for c in cliente.list_collections()]
if "documentacion_ies" in existentes:
    cliente.delete_collection("documentacion_ies")

coleccion = cliente.create_collection(
    name="documentacion_ies",
    metadata={"hnsw:space": "cosine"},
)
print("Coleccion creada:", coleccion.name)


## Guardar cinco frases

Este es el paso de cargar la base. Cada frase ya es un trozo, así que no hay que partirla.

Primero `encode` calcula las cinco listas con el modelo que acabamos de cargar. Después `add` las guarda. El orden importa: el vector que entra es el nuestro. La base no lo inventa.

La categoría (`Secretaria`, `Servicios`…) es un metadato. Sirve para filtrar, como un `WHERE`. No cambia los números del vector.


In [ ]:
frases = [
    "Procedimiento de formalizacion de matricula en el curso de especializacion.",
    "Para matricularse hay que presentar la solicitud en secretaria.",
    "El horario de la cafeteria del centro es de 8:00 a 14:00.",
    "Las pruebas de acceso a ciclos formativos se realizan en junio.",
    "El plan de orientacion ayuda al alumnado a elegir estudios.",
]
categorias = [
    {"categoria": "Oferta Educativa", "fuente": "ejemplo-oferta"},
    {"categoria": "Secretaria", "fuente": "ejemplo-acceso"},
    {"categoria": "Servicios", "fuente": "ejemplo-horarios"},
    {"categoria": "Secretaria", "fuente": "ejemplo-acceso"},
    {"categoria": "Orientacion", "fuente": "ejemplo-orientacion"},
]
ids = ["doc1", "doc2", "doc3", "doc4", "doc5"]
vectores = modelo.encode(frases).tolist()

coleccion.add(
    ids=ids,
    documents=frases,
    embeddings=vectores,
    metadatas=categorias,
)
print("Registros guardados:", coleccion.count())


## Mirar el vector que quedó dentro

Antes de preguntar, se comprueba que la base ha guardado lo que le hemos dado.

Pedimos el registro `doc1` con su texto y su vector. La longitud tiene que ser 384, la misma de la práctica 1. Si la base hubiera llamado a otro modelo a escondidas, esta comprobación fallaría.


In [ ]:
interno = coleccion.get(ids=["doc1"], include=["embeddings", "documents"])
texto = interno["documents"][0]
vector = interno["embeddings"][0]
print("Texto:", texto)
print("Dimensiones:", len(vector))
print("Diez primeros numeros:", list(vector[:10]))


## Preguntar con otras palabras

La pregunta también es un dato. Hay que convertirla en vector con el mismo modelo y comparar esa lista con las que están guardadas. `query_embeddings` hace la comparación. `n_results=2` pide solo los dos vecinos más cercanos.

La frase no contiene «formalizacion» ni «matricula». Un `LIKE` no encontraría la primera frase. La base vectorial sí puede, porque no busca las letras: busca el punto más cercano.

La distancia que imprime ChromaDB en esta colección es más pequeña cuanto más se parecen. Cerca de 0, muy parecido.


In [ ]:
pregunta = "Como me inscribo en el curso?"
vector_pregunta = modelo.encode([pregunta]).tolist()
resultados = coleccion.query(query_embeddings=vector_pregunta, n_results=2)

print("Pregunta:", pregunta)
print()
for i, doc in enumerate(resultados["documents"][0]):
    distancia = resultados["distances"][0][i]
    categoria = resultados["metadatas"][0][i]["categoria"]
    print(f"{i + 1}. distancia={distancia:.4f}  categoria={categoria}")
    print(f"   {doc}")
    print()


## Filtrar por categoría

La cercanía y el filtro no son la misma cosa.

La pregunta es amplia. Sin filtro podrían salir textos de varias categorías. `where` obliga a que la categoría sea `Secretaria`, igual que un `WHERE` en SQL. Eso no crea otro vector: quita registros y, entre los que quedan, ordena por cercanía.

Si el filtro se aplicara después de pedir los dos más cercanos, se podrían perder los de Secretaría. Por eso va dentro de la misma consulta.


In [ ]:
pregunta = "Cuando hay que hacer los tramites?"
vector_pregunta = modelo.encode([pregunta]).tolist()
filtrados = coleccion.query(
    query_embeddings=vector_pregunta,
    n_results=2,
    where={"categoria": "Secretaria"},
)
print("Pregunta:", pregunta)
print("Filtro: categoria = Secretaria")
print()
for doc, meta in zip(filtrados["documents"][0], filtrados["metadatas"][0]):
    print("-", doc, f"({meta['categoria']})")


## Memoria y disco

La colección de arriba es un cálculo en memoria. Al cerrar Colab desaparece. Una base de datos tiene que seguir ahí mañana.

`PersistentClient` escribe una carpeta. Se guardan los mismos cinco registros, se cierra el cliente y se abre otro apuntando a la misma carpeta. Si el recuento sigue siendo 5, los vectores han sobrevivido fuera de la variable.

En el proyecto la carpeta no estará en Colab: estará en el servidor. La idea es la misma.


In [ ]:
from pathlib import Path

ruta = Path("/content/chroma_ies")
cliente_disco = chromadb.PersistentClient(path=str(ruta))
if "documentacion_ies" in [c.name for c in cliente_disco.list_collections()]:
    cliente_disco.delete_collection("documentacion_ies")

en_disco = cliente_disco.create_collection(
    name="documentacion_ies",
    metadata={"hnsw:space": "cosine"},
)
en_disco.add(ids=ids, documents=frases, embeddings=vectores, metadatas=categorias)

otro = chromadb.PersistentClient(path=str(ruta))
abierta = otro.get_collection("documentacion_ies")
print("Registros al volver a abrir la carpeta:", abierta.count())
print("Carpeta:", ruta)


## Ejercicio

1. Añade el texto «El centro dispone de aparcamiento para bicicletas.» con el id `doc6` y la categoría `Servicios`.
2. Calcula tú el vector con `modelo.encode`. No dejes que ChromaDB lo invente.
3. Pregunta «Donde puedo dejar la bicicleta?» y pide un solo resultado.
4. Imprime el texto y la distancia.

La distancia tiene que ser pequeña, y la categoría tiene que ser `Servicios`.


In [ ]:
# 1. Calcula el vector de la frase nueva.
# 2. coleccion.add(...) o en_disco.add(...)
# 3. Pregunta y muestra la distancia.
